# RNN: Mortality Prediction on MIMIC-III (With code_mapping)

This notebook runs the RNN model for mortality prediction **with** `code_mapping` enabled.
Raw codes are mapped to grouped vocabularies before building the embedding table:
- ICD9CM → CCSCM (diagnosis codes → CCS categories)
- ICD9PROC → CCSPROC (procedure codes → CCS categories)
- NDC → ATC (drug codes → ATC categories)

**Model:** RNN (GRU)  
**Task:** In-hospital mortality prediction  
**Dataset:** Synthetic MIMIC-III (`dev=False`)

## Step 1: Load the MIMIC-III Dataset

In [1]:
!pip install --force-reinstall git+https://github.com/lookman-olowo/PyHealth.git@feature/code-mapping
!pip install ipywidgets

# ! pip uninstall pyhealth -y

  Cloning https://github.com/lookman-olowo/PyHealth.git (to revision feature/code-mapping) to /tmp/pip-req-build-22fn3kzk
  Running command git clone --filter=blob:none --quiet https://github.com/lookman-olowo/PyHealth.git /tmp/pip-req-build-22fn3kzk
  Running command git checkout -b feature/code-mapping --track origin/feature/code-mapping
  Switched to a new branch 'feature/code-mapping'
  branch 'feature/code-mapping' set up to track 'origin/feature/code-mapping'.
  Resolved https://github.com/lookman-olowo/PyHealth.git to commit ec2002f08f21a85c053b7ae7ec495dadbf741c49
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached accelerate-1.12.0-py3-none-any.whl.metadata (19 kB)
  Using cached dask-2025.11.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
  Using cached linear_attention_transformer-0.19.1-py3-none-any.whl.metadata (787 bytes)
  U

In [2]:
import tempfile

from pyhealth.datasets import MIMIC3Dataset

base_dataset = MIMIC3Dataset(
    root="/home/lolowo2",
    tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS"],
    cache_dir=tempfile.TemporaryDirectory().name,
    dev=False,
)

base_dataset.stats()

No config path provided, using default config
Initializing mimic3 dataset from /home/lolowo2 (dev mode: False)
Using provided cache_dir: /tmp/tmp5gf_ocv4/4f338cfd-b388-50e8-9d9c-fa4872e51b6c
No cached event dataframe found. Creating: /tmp/tmp5gf_ocv4/4f338cfd-b388-50e8-9d9c-fa4872e51b6c/global_event_df.parquet
Scanning table: patients from /home/lolowo2/PATIENTS.csv.gz
Scanning table: admissions from /home/lolowo2/ADMISSIONS.csv.gz
Scanning table: icustays from /home/lolowo2/ICUSTAYS.csv.gz
Scanning table: diagnoses_icd from /home/lolowo2/DIAGNOSES_ICD.csv.gz
Joining with table: /home/lolowo2/ADMISSIONS.csv.gz
Scanning table: procedures_icd from /home/lolowo2/PROCEDURES_ICD.csv.gz
Joining with table: /home/lolowo2/ADMISSIONS.csv.gz
Scanning table: prescriptions from /home/lolowo2/PRESCRIPTIONS.csv.gz
Joining with table: /home/lolowo2/ADMISSIONS.csv.gz
Caching event dataframe to /tmp/tmp5gf_ocv4/4f338cfd-b388-50e8-9d9c-fa4872e51b6c/global_event_df.parquet...
Dataset: mimic3
Dev mode: Fa

## Step 2: Define the Mortality Prediction Task

We override the task's `input_schema` to enable `code_mapping` on each sequence feature.
This is the **only difference** from the baseline notebook.

In [3]:
from pyhealth.tasks import MortalityPredictionMIMIC3

task = MortalityPredictionMIMIC3()

# Enable code_mapping to collapse granular codes into grouped vocabularies
task.input_schema = {
    "conditions": ("sequence", {"code_mapping": ("ICD9CM", "CCSCM")}),
    "procedures": ("sequence", {"code_mapping": ("ICD9PROC", "CCSPROC")}),
    "drugs": ("sequence", {"code_mapping": ("NDC", "ATC")}),
}

samples = base_dataset.set_task(task)

print(f"Generated {len(samples)} samples")
print(f"\nInput schema: {samples.input_schema}")
print(f"Output schema: {samples.output_schema}")

Setting task MortalityPredictionMIMIC3 for mimic3 base dataset...
Task cache paths: task_df=/tmp/tmp5gf_ocv4/4f338cfd-b388-50e8-9d9c-fa4872e51b6c/tasks/MortalityPredictionMIMIC3_c67969dc-13b3-5ab7-977f-60956867cc5d/task_df.ld, samples=/tmp/tmp5gf_ocv4/4f338cfd-b388-50e8-9d9c-fa4872e51b6c/tasks/MortalityPredictionMIMIC3_c67969dc-13b3-5ab7-977f-60956867cc5d/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Applying task transformations on data with 1 workers...
Detected Jupyter notebook environment, setting num_workers to 1
Single worker mode, processing sequentially
Worker 0 started processing 46520 patients. (Polars threads: 16)


  0%|          | 0/46520 [00:00<?, ?it/s]

Rank 0 inferred the following `['bytes']` data format.


100%|██████████| 46520/46520 [01:19<00:00, 582.37it/s]

Worker 0 finished processing patients.
Fitting processors on the dataset...


...

  0%|          | 0/9583 [00:00<?, ?it/s]

Rank 0 inferred the following `['str', 'str', 'no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:18', 'no_header_tensor:1']` data format.


100%|██████████| 9583/9583 [00:02<00:00, 4603.50it/s]

Worker 0 finished processing samples.


Cached processed samples to /tmp/tmp5gf_ocv4/4f338cfd-b388-50e8-9d9c-fa4872e51b6c/tasks/MortalityPredictionMIMIC3_c67969dc-13b3-5ab7-977f-60956867cc5d/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Generated 9583 samples

Input schema: {'conditions': ('sequence', {'code_mapping': ('ICD9CM', 'CCSCM')}), 'procedures': ('sequence', {'code_mapping': ('ICD9PROC', 'CCSPROC')}), 'drugs': ('sequence', {'code_mapping': ('NDC', 'ATC')})}
Output schema: {'mortality': 'binary'}


## Step 3: Dataset Statistics

In [4]:
print("Sample structure:")
print(samples[0])

print("\n" + "=" * 50)
print("Processor Vocabulary Sizes:")
print("=" * 50)
for key, proc in samples.input_processors.items():
    if hasattr(proc, 'code_vocab'):
        print(f"{key}: {len(proc.code_vocab)} codes (including <pad>, <unk>)")

mortality_count = sum(float(s.get("mortality", 0)) for s in samples)
print(f"\nTotal samples: {len(samples)}")
print(f"Mortality rate: {mortality_count / len(samples) * 100:.2f}%")
print(f"Positive samples: {int(mortality_count)}")
print(f"Negative samples: {len(samples) - int(mortality_count)}")

Sample structure:
{'hadm_id': '164713', 'patient_id': '10004', 'conditions': tensor([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16]), 'procedures': tensor([2, 3, 4, 5, 6, 7, 8]), 'drugs': tensor([ 2,  3,  4,  5,  6,  7,  3,  8,  9, 10, 11, 12,  6, 13, 14, 15, 15,  3,
         3,  3, 16, 17, 18, 19,  8, 20, 21,  3, 21, 22,  3,  4, 23, 24,  3, 25,
        25, 26, 27,  3, 15, 26, 26, 26, 28, 29, 28, 30,  5, 31, 32, 30, 33, 33,
        34, 35, 11, 33, 36, 35,  8,  8, 35,  4,  4,  9, 10, 35, 35, 37, 35, 14,
        17, 38, 39,  4,  4, 12,  3, 10, 35, 28, 28,  6, 29, 28, 28, 40, 41, 38,
        19,  4, 38, 38,  6,  4, 42, 10, 43,  4, 43, 44,  3, 45, 43, 43, 43,  4,
         9, 35, 46, 45, 43,  6, 47,  6, 48, 49, 50, 51, 52,  6,  4,  9, 38, 53,
         3,  6,  6,  6,  4, 50, 41, 54,  6,  6,  6,  6,  6,  6, 47, 55, 56, 51,
        57,  6,  6,  6,  6,  6,  6, 35, 35, 53, 57,  6,  6,  6, 35, 19, 58, 35,
        59, 59, 59,  6,  6,  6, 47, 54, 53, 43, 42, 29, 28, 28, 60, 31, 61, 11,

## Step 4: Split the Dataset

In [5]:
from pyhealth.datasets import split_by_patient

train_dataset, val_dataset, test_dataset = split_by_patient(
    samples, [0.8, 0.1, 0.1], seed=42
)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Training samples: 7713
Validation samples: 909
Test samples: 961


## Step 5: Create Data Loaders

In [6]:
from pyhealth.datasets import get_dataloader

train_dataloader = get_dataloader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = get_dataloader(val_dataset, batch_size=32, shuffle=False)
test_dataloader = get_dataloader(test_dataset, batch_size=32, shuffle=False)

print(f"Training batches: {len(train_dataloader)}")
print(f"Validation batches: {len(val_dataloader)}")
print(f"Test batches: {len(test_dataloader)}")

Training batches: 242
Validation batches: 29
Test batches: 31


## Step 6: Initialize the RNN Model

In [7]:
from pyhealth.models import RNN

model = RNN(
    dataset=samples,
    embedding_dim=128,
    hidden_dim=128,
)

print(f"Model initialized with {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"\nModel architecture:")
print(model)

Model initialized with 693,889 parameters

Model architecture:
RNN(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (conditions): Embedding(268, 128, padding_idx=0)
    (procedures): Embedding(204, 128, padding_idx=0)
    (drugs): Embedding(2624, 128, padding_idx=0)
  ))
  (rnn): ModuleDict(
    (conditions): RNNLayer(
      (dropout_layer): Dropout(p=0.5, inplace=False)
      (rnn): GRU(128, 128, batch_first=True)
    )
    (procedures): RNNLayer(
      (dropout_layer): Dropout(p=0.5, inplace=False)
      (rnn): GRU(128, 128, batch_first=True)
    )
    (drugs): RNNLayer(
      (dropout_layer): Dropout(p=0.5, inplace=False)
      (rnn): GRU(128, 128, batch_first=True)
    )
  )
  (fc): Linear(in_features=384, out_features=1, bias=True)
)


## Step 7: Train the Model

In [8]:
from pyhealth.trainer import Trainer

trainer = Trainer(
    model=model,
    metrics=["roc_auc", "pr_auc", "accuracy", "f1"],
)

trainer.train(
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    epochs=50,
    monitor="roc_auc",
    optimizer_params={"lr": 1e-3},
)

RNN(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (conditions): Embedding(268, 128, padding_idx=0)
    (procedures): Embedding(204, 128, padding_idx=0)
    (drugs): Embedding(2624, 128, padding_idx=0)
  ))
  (rnn): ModuleDict(
    (conditions): RNNLayer(
      (dropout_layer): Dropout(p=0.5, inplace=False)
      (rnn): GRU(128, 128, batch_first=True)
    )
    (procedures): RNNLayer(
      (dropout_layer): Dropout(p=0.5, inplace=False)
      (rnn): GRU(128, 128, batch_first=True)
    )
    (drugs): RNNLayer(
      (dropout_layer): Dropout(p=0.5, inplace=False)
      (rnn): GRU(128, 128, batch_first=True)
    )
  )
  (fc): Linear(in_features=384, out_features=1, bias=True)
)
Metrics: ['roc_auc', 'pr_auc', 'accuracy', 'f1']
Device: cuda

Training:
Batch size: 32
Optimizer: <class 'torch.optim.adam.Adam'>
Optimizer params: {'lr': 0.001}
Weight decay: 0.0
Max grad norm: None
Val dataloader: <torch.utils.data.dataloader.DataLoader object at 0x7adf35c69310>
Monitor: r

Epoch 0 / 50:   0%|          | 0/242 [00:00<?, ?it/s]

--- Train epoch-0, step-242 ---
loss: 0.3839


Evaluation: 100%|██████████| 29/29 [00:02<00:00, 11.21it/s]

--- Eval epoch-0, step-242 ---
roc_auc: 0.6327
pr_auc: 0.2036
accuracy: 0.8856
f1: 0.0000
loss: 0.3565
New best roc_auc score (0.6327) at epoch-0, step-242



...

--- Eval epoch-48, step-11858 ---
roc_auc: 0.5769
pr_auc: 0.1593
accuracy: 0.8504
f1: 0.1053
loss: 1.1049



Epoch 49 / 50:   0%|          | 0/242 [00:00<?, ?it/s]

--- Train epoch-49, step-12100 ---
loss: 0.0265


Evaluation: 100%|██████████| 29/29 [00:02<00:00, 11.24it/s]

--- Eval epoch-49, step-12100 ---
roc_auc: 0.5634
pr_auc: 0.1611
accuracy: 0.8460
f1: 0.1026
loss: 1.1359
Loaded best model


## Step 8: Evaluate on Test Set

In [10]:
test_results = trainer.evaluate(test_dataloader)

print("\n" + "=" * 50)
print("Test Set Performance (WITH code_mapping)")
print("=" * 50)
for metric, value in test_results.items():
    print(f"{metric}: {value:.4f}")

Evaluation: 100%|██████████| 31/31 [00:02<00:00, 11.59it/s]


Test Set Performance (WITH code_mapping)
roc_auc: 0.6026
pr_auc: 0.1883
accuracy: 0.8793
f1: 0.0938
loss: 0.3561
